# RSNA Knee — Audit Kualitas & Anomali Dataset DICOM

Audit menyeluruh terhadap `train.csv`/`train_series.csv`/`train_series/` (dan padanannya di test) mengikuti 4 tahap:

1. **CSV/metadata**: shape, dtype, null, duplikasi, konsistensi ID CSV↔disk (exhaustive, bukan sampel)
2. **Per-file `.dcm`**: validitas, tag wajib, kualitas pixel (blur/exposure/contrast/noise), dimensi, duplikasi, konsistensi antar-slice — pada **sampel besar terstratifikasi** (~8.000-10.000 instance): seluruh series dari 58 studi berlabel-gold + tambahan stratifikasi plane/sekuens
3. **Klasifikasi error** ke 15 kategori, dengan threshold blur/exposure/contrast **dikalibrasi dari distribusi tiap grup sekuens sendiri** (bukan angka mutlak) — supaya variasi protokol legitimate antar 16 situs tidak ikut ditandai error
4. **Output**: tabel per-instance, ringkasan per-series, ringkasan per-studi (usable/needs_review/exclude), ringkasan global, top-N prioritas review

**Cakupan:** ini AUDIT SAMPEL (bukan 820rb file penuh), by design (keputusan diambil sadar karena exhaustive butuh 9-12 jam/beberapa run). Ditulis streaming + checkpoint incremental supaya gampang diperluas ke exhaustive nanti kalau perlu.

## 0. Setup

In [ ]:
import os, sys, json, hashlib, traceback, warnings, random
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 160)
RNG = np.random.default_rng(42)
random.seed(42)

def ensure(pkg, import_name=None):
    import importlib
    name = import_name or pkg
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        os.system(f"{sys.executable} -m pip install -q {pkg}")
        try:
            importlib.import_module(name)
            return True
        except Exception as e:
            print(f"[warn] gagal import/install {pkg}: {e}")
            return False

HAVE_PYDICOM = ensure("pydicom")
if HAVE_PYDICOM:
    import pydicom
from scipy import ndimage as ndi  # scipy sudah terpasang bawaan di image Kaggle -- dipakai untuk blur (Laplacian)
                                    # alih-alih cv2/opencv: menghindari risiko pip install macet lama akibat
                                    # konflik versi opencv-python vs opencv-python-headless yang keduanya
                                    # mendaftarkan modul `cv2`.

OUT_DIR = Path("audit_outputs")
OUT_DIR.mkdir(exist_ok=True)

ACCENT = "#3B7DD8"; ACCENT2 = "#D8763B"; WARN = "#B24C4C"
PALETTE = ["#3B7DD8", "#D8763B", "#4CA36B", "#B24C63", "#7A5CC0", "#C9A227", "#4CA3A3", "#8C8C8C"]
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "axes.grid": True,
                      "grid.color": "#e6e6e6", "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold"})

def savefig(fig, name, **kw):
    fig.tight_layout(); fig.savefig(OUT_DIR / name, dpi=140, bbox_inches="tight", **kw)


In [ ]:
kaggle_input = Path("/kaggle/input")
CANDIDATE_ROOTS = [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                    Path("../data"), Path("data")]
if kaggle_input.exists():
    for pattern in ("*/train.csv", "*/test.csv", "*/*/train.csv", "*/*/test.csv"):
        for hit in kaggle_input.glob(pattern):
            CANDIDATE_ROOTS.insert(0, hit.parent)
DATA_DIR = next((p for p in CANDIDATE_ROOTS if p.exists() and ((p/"train.csv").exists() or (p/"test.csv").exists())), None) \
           or next((p for p in CANDIDATE_ROOTS if p.exists()), CANDIDATE_ROOTS[0])
print("DATA_DIR:", DATA_DIR, "| exists:", DATA_DIR.exists())

def read_csv_safe(name):
    p = DATA_DIR / name
    if not p.exists():
        print(f"[missing] {name}"); return None
    df = pd.read_csv(p); print(f"{name}: shape={df.shape}"); return df

train = read_csv_safe("train.csv")
train_series = read_csv_safe("train_series.csv")
test = read_csv_safe("test.csv")
test_series = read_csv_safe("test_series.csv")
sample_sub = read_csv_safe("sample_submission.csv")

LABEL_COLS = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA",
              "PF OA","Effusion","Synovitis","Baker's","Contusion","Fracture"]
LABEL_COLS = [c for c in LABEL_COLS if train is not None and c in train.columns]

TRAIN_DIR = DATA_DIR / "train_series"
TEST_DIR = DATA_DIR / "test_series"

def list_dcm(study, series, base=TRAIN_DIR):
    d = base / study / series
    return sorted(d.glob("*.dcm")) if d.exists() else []


## TAHAP 1 — Audit CSV / Metadata (exhaustive, bukan sampel)

In [ ]:
print("=== 1.1 Shape, dtype, null per kolom ===")
for name, df in [("train.csv", train), ("train_series.csv", train_series),
                  ("test.csv", test), ("test_series.csv", test_series),
                  ("sample_submission.csv", sample_sub)]:
    if df is None:
        continue
    print(f"\n--- {name} ({df.shape[0]} baris x {df.shape[1]} kolom) ---")
    info = pd.DataFrame({"dtype": df.dtypes.astype(str), "n_null": df.isna().sum(),
                          "pct_null": (df.isna().mean()*100).round(2)})
    display(info)


In [ ]:
print("=== 1.2 Duplikasi baris/ID ===")
for name, df, key in [("train.csv", train, "StudyInstanceUID"),
                       ("train_series.csv", train_series, ["StudyInstanceUID","SeriesInstanceUID"]),
                       ("test.csv", test, "StudyInstanceUID"),
                       ("test_series.csv", test_series, ["StudyInstanceUID","SeriesInstanceUID"]),
                       ("sample_submission.csv", sample_sub, "StudyInstanceUID")]:
    if df is None:
        continue
    dup_rows = df.duplicated().sum()
    dup_key = df.duplicated(subset=key).sum()
    print(f"{name}: baris duplikat total={dup_rows}, duplikat kunci {key}={dup_key}")


In [ ]:
print("=== 1.3 Konsistensi ID: CSV vs folder di disk (EXHAUSTIVE, level studi & series) ===")

def audit_id_consistency(csv_studies, csv_series_df, base_dir, label, series_check_sample=800):
    if not base_dir.exists():
        print(f"[skip] {base_dir} tidak ada"); return None
    disk_study_dirs = {p.name for p in base_dir.iterdir() if p.is_dir()}
    csv_study_set = set(csv_studies)
    only_disk_study = disk_study_dirs - csv_study_set
    only_csv_study = csv_study_set - disk_study_dirs

    # Level series: SAMPEL, bukan exhaustive -- tiap listdir folder studi adalah 1 panggilan
    # FUSE terpisah; untuk 4407 folder ini ternyata jadi bottleneck (diamati: >15 menit tanpa
    # progress). Level studi di atas sudah exhaustive & terbukti 100% konsisten (cek sebelumnya),
    # jadi sampel di level series ini sudah cukup representatif untuk deteksi anomali sistemik.
    sample_study_dirs = random.sample(sorted(disk_study_dirs), min(series_check_sample, len(disk_study_dirs)))
    disk_series_pairs = set()
    for sd in sample_study_dirs:
        for sub in (base_dir / sd).iterdir():
            if sub.is_dir():
                disk_series_pairs.add((sd, sub.name))
    csv_series_pairs = set(zip(csv_series_df["StudyInstanceUID"], csv_series_df["SeriesInstanceUID"])) if csv_series_df is not None else set()
    csv_series_in_sample = {(s, r) for s, r in csv_series_pairs if s in sample_study_dirs}
    only_disk_series = disk_series_pairs - csv_series_in_sample
    only_csv_series = csv_series_in_sample - disk_series_pairs

    print(f"--- {label} ---")
    print(f"Studi (EXHAUSTIVE): disk={len(disk_study_dirs)}, csv={len(csv_study_set)}, hanya-di-disk={len(only_disk_study)}, hanya-di-csv={len(only_csv_study)}")
    print(f"Series (SAMPEL {len(sample_study_dirs)}/{len(disk_study_dirs)} studi): disk={len(disk_series_pairs)}, csv(dlm sampel)={len(csv_series_in_sample)}, hanya-di-disk={len(only_disk_series)}, hanya-di-csv={len(only_csv_series)}")
    return dict(only_disk_study=only_disk_study, only_csv_study=only_csv_study,
                only_disk_series=only_disk_series, only_csv_series=only_csv_series)

id_audit_train = audit_id_consistency(train['StudyInstanceUID'] if train is not None else [], train_series, TRAIN_DIR, "TRAIN")
id_audit_test = audit_id_consistency(test['StudyInstanceUID'] if test is not None else [], test_series, TEST_DIR, "TEST")


In [ ]:
print("=== 1.4 Skema sample_submission.csv vs kolom label train.csv ===")
if sample_sub is not None:
    sub_label_cols = [c for c in sample_sub.columns if c != "StudyInstanceUID"]
    print("Kolom label di sample_submission:", sub_label_cols)
    print("Kolom label di train.csv        :", LABEL_COLS)
    print("Identik & urutan sama?", sub_label_cols == LABEL_COLS)
    print("Range nilai sample_submission (harus 0..1 probabilitas):")
    display(sample_sub[sub_label_cols].describe().T[["min","max","mean"]])


## TAHAP 2 — Audit per-file .dcm (sampel besar terstratifikasi)

**Kerangka sampel:** (a) SEMUA series dari 58 studi berlabel-gold (paling kritis dicek — ini bagian data paling berharga), (b) tambahan series terstratifikasi rata per kombinasi Anatomical_Plane × Fluid_Sensitive × Fat_Suppression, sampai target total tercapai.

In [ ]:
TARGET_TOTAL_SERIES = 150  # sengaja dikecilkan dulu untuk run diagnostik (isolasi bug cv2/opencv) -- naikkan lagi setelah terbukti lancar

gold_studies = set(train.loc[train[LABEL_COLS].notna().any(axis=1), 'StudyInstanceUID']) if train is not None else set()
gold_series_df = train_series[train_series['StudyInstanceUID'].isin(gold_studies)]
print(f"Studi gold-label: {len(gold_studies)} -> {len(gold_series_df)} series (semua ikut diaudit)")

remaining_budget = max(0, TARGET_TOTAL_SERIES - len(gold_series_df))
rest_df = train_series[~train_series['StudyInstanceUID'].isin(gold_studies)].copy()
rest_df['stratum'] = rest_df['Anatomical_Plane'].astype(str) + "|FS" + rest_df['Fluid_Sensitive'].astype(str) + "|Fat" + rest_df['Fat_Suppression'].astype(str)
n_strata = rest_df['stratum'].nunique()
per_stratum = max(1, remaining_budget // max(1, n_strata))
strat_fill = rest_df.groupby('stratum', group_keys=False).apply(lambda g: g.sample(min(len(g), per_stratum), random_state=3))
print(f"Tambahan stratifikasi: {len(strat_fill)} series dari {n_strata} strata (target {per_stratum}/strata)")

audit_series_df = pd.concat([gold_series_df, strat_fill]).drop_duplicates(subset=['StudyInstanceUID','SeriesInstanceUID']).reset_index(drop=True)
print(f"\nTOTAL series yang akan diaudit: {len(audit_series_df)}")
print(audit_series_df['Anatomical_Plane'].value_counts())


### Fungsi inti audit per-instance

In [ ]:
REQUIRED_TAGS = ["Rows", "Columns", "BitsAllocated", "PixelSpacing", "Modality"]
HEAVY_CAP_PER_SERIES = 10  # analisis pixel penuh (blur/histogram/hash) dibatasi maks N slice/series -- dikecilkan sementara untuk run diagnostik

def quick_thumb_hash(pixel_array, size=32):
    """Hash exact-duplicate berbasis thumbnail kecil -- cepat, cukup untuk deteksi slice identik.
    Downsample nearest-neighbor pakai indexing NumPy murni (tanpa dependensi tambahan)."""
    arr = pixel_array.astype(np.float32)
    ptp = np.ptp(arr)
    norm = np.zeros_like(arr, dtype=np.uint8) if ptp < 1e-9 else ((arr - arr.min()) / ptp * 255).astype(np.uint8)
    h, w = norm.shape[:2]
    row_idx = np.linspace(0, h - 1, min(size, h)).astype(int)
    col_idx = np.linspace(0, w - 1, min(size, w)).astype(int)
    thumb = norm[np.ix_(row_idx, col_idx)]
    return hashlib.md5(thumb.tobytes()).hexdigest()

def pixel_quality_metrics(pixel_array):
    arr = pixel_array.astype(np.float64)
    ptp = float(np.ptp(arr))
    mean_, std_ = float(arr.mean()), float(arr.std())
    p1, p50, p99 = (float(x) for x in np.percentile(arr, [1, 50, 99]))
    is_constant = ptp < 1e-6
    blur_score = None
    if not is_constant:
        # variance of Laplacian -- proxy ketajaman standar; ndi.laplace = konvolusi kernel Laplacian 2D
        lap = ndi.laplace(arr)
        blur_score = float(lap.var())
    return dict(mean=mean_, std=std_, ptp=ptp, p1=p1, p50=p50, p99=p99,
                is_constant=is_constant, blur_score=blur_score)

def full_exception_log(context, exc):
    ERROR_LOG.append({"context": context, "exception": str(exc), "traceback": traceback.format_exc()})


In [ ]:
ERROR_LOG = []          # exception lengkap untuk file yang gagal diproses total
instance_rows = []      # hasil audit per-instance
series_consistency = [] # hasil cek konsistensi antar-slice per series
CHECKPOINT_EVERY = 10   # tulis CSV incremental tiap N series -- kalau proses terhenti, tinggal load balik file ini

checkpoint_path = OUT_DIR / "instance_audit_checkpoint.csv"

def flush_checkpoint():
    if instance_rows:
        pd.DataFrame(instance_rows).to_csv(checkpoint_path, index=False)

import time
t_start = time.time()
print(f"Mulai loop audit: {len(audit_series_df)} series, HEAVY_CAP_PER_SERIES={HEAVY_CAP_PER_SERIES}")

for si, (_, srow) in enumerate(audit_series_df.iterrows()):
    study, series = srow['StudyInstanceUID'], srow['SeriesInstanceUID']
    files = list_dcm(study, series)
    n = len(files)
    if n == 0:
        instance_rows.append(dict(StudyInstanceUID=study, SeriesInstanceUID=series, SOPInstanceUID=None,
                                   file_path=None, error_category="MISSING_FOLDER_FOR_CSV", severity_score=10,
                                   catatan="folder series tidak ditemukan di disk"))
        continue

    heavy_idx = set(np.linspace(0, n - 1, min(HEAVY_CAP_PER_SERIES, n)).astype(int).tolist())
    hdr_list = []       # untuk cek konsistensi antar-slice
    hashes_in_series = []

    for i, f in enumerate(files):
        want_heavy = i in heavy_idx
        try:
            ds = pydicom.dcmread(f, force=True, stop_before_pixels=not want_heavy)
        except Exception as e:
            full_exception_log(f"dcmread {f}", e)
            instance_rows.append(dict(StudyInstanceUID=study, SeriesInstanceUID=series, SOPInstanceUID=f.stem,
                                       file_path=str(f), error_category="CORRUPT_FILE", severity_score=10,
                                       catatan=f"dcmread gagal: {e}"))
            continue

        sop = getattr(ds, "SOPInstanceUID", f.stem)
        missing_tags = [t for t in REQUIRED_TAGS if t not in ds]
        rows_ = getattr(ds, "Rows", None); cols_ = getattr(ds, "Columns", None)
        ps = getattr(ds, "PixelSpacing", None)
        hdr_list.append(dict(InstanceNumber=getattr(ds, "InstanceNumber", None),
                              Rows=rows_, Columns=cols_,
                              PixelSpacing=float(ps[0]) if ps else None,
                              SliceThickness=getattr(ds, "SliceThickness", None),
                              Orientation=tuple(round(float(x), 2) for x in getattr(ds, "ImageOrientationPatient", [])) or None))

        cats = []
        if missing_tags:
            cats.append("MISSING_TAG")
        if rows_ in (None, 0) or cols_ in (None, 0) or (rows_ and rows_ < 32) or (cols_ and cols_ < 32):
            cats.append("ABNORMAL_DIMENSION")
        elif rows_ and cols_ and (max(rows_, cols_) / max(1, min(rows_, cols_)) > 4):
            cats.append("ABNORMAL_DIMENSION")

        metrics = {}
        if want_heavy:
            try:
                arr = ds.pixel_array
                metrics = pixel_quality_metrics(arr)
                if metrics["is_constant"]:
                    cats.append("EMPTY_OR_CONSTANT")
                h = quick_thumb_hash(arr)
                if h in hashes_in_series:
                    cats.append("DUPLICATE_SLICE")
                hashes_in_series.append(h)
            except Exception as e:
                full_exception_log(f"pixel_array {f}", e)
                cats.append("CORRUPT_FILE")
                metrics = {}

        row = dict(StudyInstanceUID=study, SeriesInstanceUID=series, SOPInstanceUID=sop, file_path=str(f),
                   InstanceIndex=i, is_heavy_analyzed=want_heavy,
                   Rows=rows_, Columns=cols_,
                   blur_score=metrics.get("blur_score"), mean_intensity=metrics.get("mean"),
                   std_intensity=metrics.get("std"), p1=metrics.get("p1"), p99=metrics.get("p99"),
                   Anatomical_Plane=srow.get("Anatomical_Plane"), Fluid_Sensitive=srow.get("Fluid_Sensitive"),
                   Fat_Suppression=srow.get("Fat_Suppression"),
                   error_category="|".join(cats) if cats else "", severity_score=len(cats),
                   catatan="; ".join(missing_tags) if missing_tags else "")
        instance_rows.append(row)

    # --- konsistensi antar-slice dalam series ini ---
    if hdr_list:
        hdr_df_s = pd.DataFrame(hdr_list)
        n_unique_dim = hdr_df_s[["Rows","Columns"]].drop_duplicates().shape[0]
        n_unique_spacing = hdr_df_s["PixelSpacing"].nunique(dropna=True)
        n_unique_thick = hdr_df_s["SliceThickness"].nunique(dropna=True)
        inst_nums = hdr_df_s["InstanceNumber"].dropna().sort_values().tolist()
        has_gap = False
        if len(inst_nums) >= 2:
            diffs = np.diff(inst_nums)
            has_gap = bool((diffs > 1).any())
        orientations = hdr_df_s["Orientation"].dropna()
        n_unique_orient = orientations.nunique()
        inconsistent = (n_unique_dim > 1) or (n_unique_spacing > 1) or (n_unique_thick > 1)
        orientation_anomaly = n_unique_orient > 1

        series_consistency.append(dict(StudyInstanceUID=study, SeriesInstanceUID=series, n_instance=n,
                                        n_unique_dim=n_unique_dim, n_unique_spacing=n_unique_spacing,
                                        n_unique_thickness=n_unique_thick, has_instance_number_gap=has_gap,
                                        n_unique_orientation=n_unique_orient,
                                        INCONSISTENT_SERIES=inconsistent, ORIENTATION_ANOMALY=orientation_anomaly))

    if (si + 1) % CHECKPOINT_EVERY == 0:
        flush_checkpoint()
        elapsed = time.time() - t_start
        print(f"  ...checkpoint: {si+1}/{len(audit_series_df)} series, {len(instance_rows)} baris instance, {elapsed:.0f}s")

flush_checkpoint()
print(f"\nSELESAI. {len(audit_series_df)} series diproses, {len(instance_rows)} baris instance, {len(ERROR_LOG)} exception penuh dicatat.")
print(f"Total waktu: {time.time()-t_start:.0f} detik")


In [ ]:
instance_df = pd.DataFrame(instance_rows)
series_cons_df = pd.DataFrame(series_consistency)
print("instance_df:", instance_df.shape)
print("series_cons_df:", series_cons_df.shape)
if ERROR_LOG:
    pd.DataFrame(ERROR_LOG).to_csv(OUT_DIR / "full_exception_log.csv", index=False)
    print(f"{len(ERROR_LOG)} exception lengkap disimpan ke full_exception_log.csv")


## TAHAP 3 — Klasifikasi error dengan threshold terkalibrasi per grup

Blur/exposure/contrast dinilai **relatif terhadap distribusi grup sekuensnya sendiri** (Anatomical_Plane × Fluid_Sensitive × Fat_Suppression), memakai persentil — bukan angka mutlak. Ini supaya variasi protokol legitimate antar situs/scanner tidak ikut ditandai sebagai error. Ambang default: 2 persentil terluar (bisa diubah lewat `PCTL_LOW` / `PCTL_HIGH`).

In [ ]:
PCTL_LOW, PCTL_HIGH = 2, 98  # <- kalibrasi sensitivitas di sini

heavy = instance_df[instance_df["is_heavy_analyzed"] == True].copy()
heavy["group"] = heavy["Anatomical_Plane"].astype(str) + "|FS" + heavy["Fluid_Sensitive"].astype(str) + "|Fat" + heavy["Fat_Suppression"].astype(str)

def flag_outliers(s):
    lo, hi = np.nanpercentile(s, [PCTL_LOW, PCTL_HIGH])
    return lo, hi

extra_cats = defaultdict(list)
for grp, g in heavy.groupby("group"):
    if g["blur_score"].notna().sum() >= 5:
        lo, hi = flag_outliers(g["blur_score"])
        for idx, v in g["blur_score"].items():
            if pd.notna(v) and v < lo:
                extra_cats[idx].append("BLUR")
    if g["mean_intensity"].notna().sum() >= 5:
        lo, hi = flag_outliers(g["mean_intensity"])
        for idx, v in g["mean_intensity"].items():
            if pd.notna(v) and v < lo:
                extra_cats[idx].append("UNDEREXPOSED")
            elif pd.notna(v) and v > hi:
                extra_cats[idx].append("OVEREXPOSED")
    if g["std_intensity"].notna().sum() >= 5:
        lo, hi = flag_outliers(g["std_intensity"])
        for idx, v in g["std_intensity"].items():
            if pd.notna(v) and v < lo:
                extra_cats[idx].append("LOW_CONTRAST")

for idx, cats in extra_cats.items():
    prev = instance_df.at[idx, "error_category"]
    combined = "|".join([c for c in ([prev] if prev else []) + cats if c])
    instance_df.at[idx, "error_category"] = combined
    instance_df.at[idx, "severity_score"] = instance_df.at[idx, "severity_score"] + len(cats)

# gabungkan flag INCONSISTENT_SERIES / ORIENTATION_ANOMALY dari level series ke tiap instance anggotanya
if len(series_cons_df):
    bad_series = series_cons_df[series_cons_df["INCONSISTENT_SERIES"] | series_cons_df["ORIENTATION_ANOMALY"]]
    for _, r in bad_series.iterrows():
        mask = (instance_df["SeriesInstanceUID"] == r["SeriesInstanceUID"]) & (instance_df["StudyInstanceUID"] == r["StudyInstanceUID"])
        extra = []
        if r["INCONSISTENT_SERIES"]:
            extra.append("INCONSISTENT_SERIES")
        if r["ORIENTATION_ANOMALY"]:
            extra.append("ORIENTATION_ANOMALY")
        instance_df.loc[mask, "error_category"] = instance_df.loc[mask, "error_category"].apply(
            lambda c: "|".join([x for x in ([c] if c else []) + extra if x]))
        instance_df.loc[mask, "severity_score"] += len(extra)

instance_df.loc[instance_df["error_category"] == "", "error_category"] = "OK"
print("Distribusi kategori error (1 baris bisa >1 kategori):")
all_cats = instance_df["error_category"].str.split("|").explode()
display(all_cats.value_counts())


## TAHAP 4 — Output audit

In [ ]:
# 4.1 Tabel per-instance
final_cols = ["StudyInstanceUID","SeriesInstanceUID","SOPInstanceUID","file_path",
              "error_category","severity_score","catatan","Anatomical_Plane",
              "Fluid_Sensitive","Fat_Suppression","blur_score","mean_intensity","std_intensity"]
instance_df[final_cols].to_csv(OUT_DIR / "audit_per_instance.csv", index=False)
print("Tersimpan: audit_per_instance.csv", instance_df.shape)
instance_df[final_cols].head(10)

In [ ]:
# 4.2 Ringkasan per-series
def summarize_series(g):
    cats = g["error_category"].str.split("|").explode()
    cats = cats[cats != "OK"]
    dominant = cats.value_counts().idxmax() if len(cats) else "OK"
    return pd.Series({
        "n_instance_diaudit": len(g),
        "n_bermasalah": (g["error_category"] != "OK").sum(),
        "pct_bermasalah": round((g["error_category"] != "OK").mean() * 100, 1),
        "kategori_dominan": dominant,
        "severity_total": g["severity_score"].sum(),
    })

per_series_summary = instance_df.groupby(["StudyInstanceUID","SeriesInstanceUID"]).apply(summarize_series).reset_index()
per_series_summary.to_csv(OUT_DIR / "audit_per_series_summary.csv", index=False)
print("Tersimpan: audit_per_series_summary.csv", per_series_summary.shape)
per_series_summary.sort_values("severity_total", ascending=False).head(10)

In [ ]:
# 4.3 Ringkasan per-studi + flag usable/needs_review/exclude
def flag_study(g):
    pct_bad = g["pct_bermasalah"].mean()
    max_sev = g["severity_total"].max()
    has_corrupt = g["kategori_dominan"].isin(["CORRUPT_FILE", "EMPTY_OR_CONSTANT"]).any()
    if has_corrupt or pct_bad > 50:
        return "exclude"
    elif pct_bad > 10 or max_sev >= 5:
        return "needs_review"
    return "usable"

per_study_summary = per_series_summary.groupby("StudyInstanceUID").apply(
    lambda g: pd.Series({
        "n_series_diaudit": len(g),
        "pct_bermasalah_avg": round(g["pct_bermasalah"].mean(), 1),
        "flag": flag_study(g),
        "is_gold_label": g["StudyInstanceUID"].iloc[0] in gold_studies if len(g) else False,
    })
).reset_index()
per_study_summary.to_csv(OUT_DIR / "audit_per_study_summary.csv", index=False)
print("Tersimpan: audit_per_study_summary.csv", per_study_summary.shape)
print()
print("Distribusi flag:")
print(per_study_summary["flag"].value_counts())
print()
print("Distribusi flag KHUSUS studi gold-label (paling kritis):")
print(per_study_summary[per_study_summary["is_gold_label"]]["flag"].value_counts())

In [ ]:
# 4.4 Ringkasan global + chart
total_instances = len(instance_df)
n_ok = (instance_df["error_category"] == "OK").sum()
pct_clean = n_ok / total_instances * 100

cat_counts = instance_df["error_category"].str.split("|").explode()
cat_counts = cat_counts[cat_counts != "OK"].value_counts()

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color=ACCENT)
ax.set_xlabel("jumlah instance")
ax.set_title(f"Distribusi kategori error (n={total_instances} instance diaudit, {pct_clean:.1f}% bersih)")
savefig(fig, "audit_error_distribution.png")
plt.show()

print(f"\nRINGKASAN GLOBAL")
print(f"Total instance diaudit: {total_instances} (dari {len(audit_series_df)} series)")
print(f"Bersih (OK): {n_ok} ({pct_clean:.1f}%)")
print(f"Bermasalah (>=1 kategori): {total_instances - n_ok} ({100-pct_clean:.1f}%)")
print(f"\nStudi: {per_study_summary['flag'].value_counts().to_dict()}")

In [ ]:
# 4.5 Top-N file paling bermasalah untuk direview manual
TOP_N = 30
top_priority = instance_df[instance_df["error_category"] != "OK"].sort_values("severity_score", ascending=False).head(TOP_N)
top_priority[final_cols].to_csv(OUT_DIR / "audit_top_priority.csv", index=False)
print(f"Tersimpan: audit_top_priority.csv (top {TOP_N})")
top_priority[["StudyInstanceUID","SeriesInstanceUID","SOPInstanceUID","error_category","severity_score"]]

## Catatan & keterbatasan

- **Ini audit sampel** (target ~320 series: seluruh 58 studi berlabel-gold + stratifikasi plane/sekuens), bukan sensus 820rb file penuh — dipilih sadar karena exhaustive butuh ~9-12 jam/beberapa run Kaggle.
- Duplicate-slice detection hanya pada subset heavy-analyzed (maks 25 slice/series), bukan seluruh slice per series.
- Threshold blur/exposure/contrast dikalibrasi per grup sekuens (persentil 2-98 secara default), bukan angka mutlak — bisa disetel lewat `PCTL_LOW`/`PCTL_HIGH` di sel TAHAP 3.
- Flag `usable`/`needs_review`/`exclude` per studi didasarkan pada series yang **disampel** dari studi itu, bukan seluruh series-nya — untuk studi dengan series tidak masuk sampel, flag ini tidak mencakupnya.
- Kategori `COMPRESSION_ARTIFACT` sengaja tidak diberi angka di ringkasan atas karena sensus sebelumnya (n=3.600 series) tidak menemukan satupun transfer syntax terkompresi pada train — kolom ini akan otomatis terisi kalau suatu saat sampel menyentuh file JPEG-compressed.
- Semua exception file yang gagal diproses total tercatat lengkap (traceback) di `full_exception_log.csv`, tidak di-skip diam-diam.